In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGHEST VALUE FILTER: Cleans the 'Contract_Amount' string (removes commas/$) 
      and casts to a Double. The Numerator evaluates contracts >= 1,000,000.
   3. EXPANDED COMMA EXCEPTION HANDLING: Protects 6 known unit names containing internal 
      commas by temporarily replacing them with '[COMMA]' so the split function doesn't 
      shred them.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand the comma-separated 
      owning_lob and lob_using lists into individual rows. Restores the '[COMMA]' back 
      to a real ',' immediately after.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax 
      ('%' || String || '%') to avoid regex failures on special characters ('&', '-').
   6. NAME BRIDGE: The mapping table stores the AU Name in the Assessable_Unit_ID 
      column. This logic bridges that String Name to the Master List's AU_Name to 
      retrieve the true Numeric BusinessID.
   7. AGGREGATING (NUMERATOR & DENOMINATOR): Counts DISTINCT EngagementNumbers per AU 
      (Denominator) and DISTINCT EngagementNumbers >= 1MM (Numerator) to remove 
      duplicates caused by the flattening process.
   8. FINAL OUTPUT: Calculates the percentage (Numerator/Denominator * 100), rounds 
      to 2 decimal places, appends a '%' sign, and outputs the strict 6 columns.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings and block blank/null strings
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Extract TP data, clean contract amount, and apply Expanded Comma Exceptions
    SELECT 
        EngagementNumber,
        
        -- =========================================================================
        -- EXPANDED EXCEPTION DICTIONARY: Protect all known LOBs with internal commas
        -- =========================================================================
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using,
        
        -- Clean Contract Amount
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flattened_LOBs AS (
    -- 4. FLATTEN RULE: Split by commas, expand into individual rows, and restore protected commas
    SELECT 
        EngagementNumber, 
        Cleaned_Amount,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Third_Party_Risk_Data
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 5. Bridge mappings to Master List and calculate Numerator/Denominator
    SELECT 
        mast.BusinessID,
        -- Total Distinct Engagements mapped to this AU
        COUNT(DISTINCT f.EngagementNumber) AS Denominator,
        -- Distinct Engagements mapped to this AU that are >= 1 Million
        COUNT(DISTINCT CASE WHEN f.Cleaned_Amount >= 1000000 THEN f.EngagementNumber END) AS Numerator
    FROM Flattened_LOBs f
    -- Safe Mapping Join
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    -- Name Bridge Join
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 6. Final Output: Strict 6-column structure anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Math: Handles division by zero safely, calculates percentage, and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - High Value Contract Traceability
   
   PURPOSE: Shows the raw Contract Amount vs Cleaned Amount, prints the full 
   comma-separated LOB strings vs the exploded chunks (with EXPANDED comma 
   protection), and displays the Master AU lookup status.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Flattened AS (
    -- Extract raw data, clean the amount, and explode the LOBs into individual rows
    SELECT 
        p.EngagementNumber,
        p.Contract_Amount AS Raw_Contract_Amount,
        TRY_CAST(REPLACE(REPLACE(TRIM(p.Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount,
        p.owning_lob AS Raw_Col_K,
        p.lob_using AS Raw_Col_L,
        
        -- Expanded Comma Dictionary
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
            
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    WHERE p.EngagementNumber IS NOT NULL
)

SELECT DISTINCT
    f.EngagementNumber,
    
    -- Contract Value Traceability
    f.Raw_Contract_Amount,
    f.Cleaned_Amount,
    CASE WHEN f.Cleaned_Amount >= 1000000 THEN 'Yes - Counts in Numerator' ELSE 'No - Denominator Only' END AS Is_High_Value,
    
    -- LOB Flattening Traceability
    f.Raw_Col_K AS Original_String_From_DB_Col_K,
    f.Raw_Col_L AS Original_String_From_DB_Col_L,
    REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Final_Restored_Chunk,
    
    -- Mapping & Bridging Traceability
    map.TPRM_Units AS Matched_Mapping_String,
    map.Assessable_Unit_ID AS Mapping_Table_AU_Name,
    mast.Master_Numeric_ID AS Bridged_Master_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'MISSING FROM MASTER LIST' ELSE 'SUCCESSFULLY BRIDGED' END AS Master_List_Status
    
FROM Flattened f
LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
-- Safe Mapping Join
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(REPLACE(TRIM(exploded_lob), '[COMMA]', ',')) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
-- Name Bridge Join
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
ORDER BY f.EngagementNumber;